## 1. 인트로 <br>
### **1-1. RNN 한계와 Attention 등장 배경** <br>
- RNN: 장기 의존성 문제, 기울기 소실 문제 -> LSTM: 초반부 데이터가 forget gate를 더 많이 통과하므로 긴 문장에서 여전히 기울기 소실 문제 발생 가능     
- Seq2Seq: 중요한 정보 손실되거나 출력 단어마다 필요한 정보를 다르게 반영하기 어려움 -> Attention 등장      
    - Attention: 중요한 정보에 더 집중하는 메커니즘 (현재 시점 가장 중요한 부분에 더 높은 가중치 부여)     
        - 출력 시점마다 새로운 문맥 벡터 동적으로 계산 = 병렬화     
        - 장점: 시간 경과에 대한 유연성, 공간에 대한 유연성, 병렬화     

### **1-2. Attention is All You Need** <br>
- RNN과 RNN 기반 모델의 한계점 -> RNN 순환 구조 완전히 제거하고 self-attention에 기반한 트랜스포머 모델 제안    
- 성공적인 모델링에 recurrence 및 concolution이 필수가 아님을 증명해 순환 구조 없이 성능 낼 수 있음 + 병렬화 가능    
- LLM 탄생의 기반, NLP 표준으로 자리 잡음, 타 분야로의 확장     

**Trnansformer** <br>
= RNN 순차 계산 방식 버리고 Attention만으로 문장의 의미와 구조 파악하는 모델   
- Self-Attention: 문장 내 단어가 다른 단어들과 얼마나 중요한 관계를 맺는지 한 번에 파악    
- Multi-Head Attention: 셀프 어텐션을 여러 개의 Head로 동시에 서로 다른 관점에서 실행      
- Positional Encoding: 단어 위치 정보를 벡터에 추가해 단어가 문장의 몇 번째 위치에 존재하는지      

## 2. Self- Attention <br>
### **2-1. Query, Key, Value 개념** <br>
- 쿼리 = 질문/요청 -> 알고 싶거나 초점을 맞추는 대상   
- 키 = 모든 정보가 달고 있는 색인 -> 키와 자신을 비교해 관련성 확인   
- 밸류 = 키와 한 쌍으로 묶인 실제 내용 -> 가장 관련성이 높은 키가 선택되면, 모델은 키에 해당하는 밸류 가져와 사용     

### **2-2. Cross Attention vs. Self-Attention 개념 및 과정** <br>

|Cross Attention|Self-Attention|    
|---------|---------|         
|하나의 시퀀서가 완전히 다른 시퀀스 참고해 정보 만들어냄|한 문장 내 모든 단어 간 관계 계산|         
|서로 다른 시퀀스|동일한 시퀀스|         
|인코더-디코더 연결, 정보 병목 해소|시퀀스 내 문맥 및 의존 관계 파악|         
|RNN 기반|트랜스포머 기반|         
|병렬 처리 불가능|병렬 처리 가능|         

### **2-3. Scaled Dot-Product Attention** <br>
- Dot-Product Attention: Attention Score 계산하는 기본 방법 중 하나로, 쿼리와 키 벡터를 내적해 유사도 계산 -> 벡터 차원 커질수록 내적 값이 지나치게 커지거나 작아져 기울기 소실 문제 발생 가능   
- Scaled Dot-Product Attention: Dot-Product Attention 진행 후 스케일링 추가     

## 3. Multi-Head Attention <br>
### **3-1. Multo-Head Attention 개념** <br>
= 문장 내 단어들 간 관계 파악을 위해 하나의 가중치 행렬만 학습하고 번역에 사용    
- 512차원의 입력 벡터를 위한 어텐션 가중치 분포 계산 후 가중합해 새로운 512차원 벡터 출력    
- 한 어텐션 분포 내 단어 간 문법 관계, 의미 관계, 위치 관계 등 모든 정보를 한 번에 담아 가중 평균하는 Single=Head Attention과 달리 한 단어왇 다른 단어 간 관계를 여러 차원으로 나눠 병렬 학습           
- 64차원씩 8개의 벡터로 나눠 스코어 병렬로 여러 번 계산    

### **3-2. Multi-Head Attention 작동 방식** <br>
**분할** <br>
- 하나의 512차원 벡터가 들어오면 8개의 작은 벡터 그룹으로 투영해 서로 다른 관점 8개로 나눔      
**병렬 어텐션 계산** <br>
- 8개의 헤드가 서로 간섭하지 않고 병렬로 Scaled 내적 어텐션 계산       
- 점수 계산 -> 크기 조절 -> 가중치 변환 (소프트맥스) -> 가중합       
**결합 및 최종 투영** <br>
- 8개 헤드가 낸 8개의 분석 결과 하나로 결합     
**결합**<br>
- 64차월 결과 벡터 * 8 = 하나의 512차원 벡터     
**최종 투영**<br>
- 헤드의 분석 결과가 나열된 상태인데, 이를 융합하고 최종 정리하기 이ㅜ해 또 다른 가중치 행렬 곱함     

## 4. Transformer 아키텍처<br>
### **4-1. 전처리 단계** <br>
**토큰화**<br>
= 입력 텍스트를 모델이 처리 가능한 단위로 나누는 첫 번째 단계     
- 대부분 토큰은 한단어나 문장 부호, 일부 단어는 하나 이상의 토큰     
**임베딩**<br>
= 토큰화된 각 단위를 숫자 벡터로 변환     
- 단어 의미를 공간적으로 표현      
- 유사한 단어는 유사한 숫자로 변환하는 덕분에 단어 문맥 포착과 관계를 파악하는 어텐션 작동 전 단어 기본 속성 제공      
**Positional Encoding**<br>
= 순서 정보를 임베딩 벡터에 추가      
- 문법이나 문맥 파악을 위해 필요한 등장 순서 / 위치 정보 제공       

### **4-2. Encoder와 Decoder** <br>
**인코더**<br>
= 입력 문장을 이해하고 요약된 의미 벡터로 변환       
- 전처리 과정 및 인코더 레이어로 구성     
- Multi-Head Attention, Feed Forward 두 가지 하위 레이어로 구성     
    - Feed-Forward Layer = 입력 벡터 차원 확장하고 비선형 변환 적용해 새로운 표현 생성하는 신경망 구조     
- 잔차 연결, 층 정규화     

**디코더**<br>
= 인코더가 분석한 입력 문장의 의미 벡터를 받아 출력 문장을 순차적으로 생성하는 역할       
- 전처리, 디코더 레이어, 선형 레이어, 소프트맥스 레이어로 구성      

**디코더 레이어**<br>
- Masked Multi-Head Attention = 미래 시점의 단어 정보 참고 못하도록 마스크 적용     
- Encoder-Decoder Attention = 디코더가 인코더의 출력을 참고하며 현재 생성 중인 단어를 입력 문장의 의미와 연결하는 과정       
- Feed-Forward Layer

### **4-3. 전체 모델 및 데이터 흐름** <br>
- 입력 준비 -> 인코더 -> 디코더    
    - 입력 준비: 문장을 의미와 위치 벡터로 바꿈 - 임베딩, 포지셔널 인코딩     
    - 인코더: 입력 문장의 문맥적 의미 이해 - 셀프 어텐션, 관계 점수 계산, 가중합, 멀티 헤드      
    - 디코더: 번역 문장을 한 단어씩 생성     